**Hybrid Filtering**

Hybrid filtering combines content-based filtering and collaborative filtering to provide more accurate and personalized recommendations. Content-based filtering recommends movies similar to those a user likes based on metadata such as genres and tags, while collaborative filtering recommends movies based on similar users’ preferences. By combining both approaches, hybrid systems reduce limitations such as cold-start problems and over-specialization, making recommendations more reliable and diverse.

**Advantages of Hybrid Filtering**



*  Combines the strengths of content-based and collaborative filtering.

* Reduces the cold-start problem for new users and new items.

* Provides more accurate and personalized recommendations.

* Avoids over-specialization by introducing diverse recommendations.

* Improves overall recommendation quality and user satisfaction.



**Limitations of Hybrid Filtering**



*  More complex to design and implement compared to single methods.

* Requires higher computational resources and processing time.

* Needs both user interaction data and item metadata.

* System tuning and weighting between models can be challenging.

* Difficult to scale efficiently for very large datasets.



In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:

movies = pd.read_csv("/movies.csv")
ratings = pd.read_csv("/ratings.csv")
tags = pd.read_csv("/tags.csv")

In [3]:
tags_grouped = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x)).reset_index()
movies_cb = pd.merge(movies, tags_grouped, on="movieId", how="left")
movies_cb["tag"] = movies_cb["tag"].fillna("")
movies_cb["content"] = movies_cb["genres"] + " " + movies_cb["tag"]

In [4]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies_cb["content"])
content_similarity = cosine_similarity(tfidf_matrix)

In [5]:
user_movie = ratings.pivot_table(index="userId", columns="movieId", values="rating").fillna(0)
user_similarity = cosine_similarity(user_movie)


In [6]:
def hybrid_recommend(user_id, movie_title, top_n=5):
    movie_idx = movies_cb[movies_cb["title"] == movie_title].index[0]
    content_scores = list(enumerate(content_similarity[movie_idx]))

    user_idx = user_movie.index.get_loc(user_id)
    collab_scores = user_similarity[user_idx]

    hybrid_scores = [
        (i, content_scores[i][1] * 0.6 + collab_scores.mean() * 0.4)
        for i in range(len(content_scores))
    ]

    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in hybrid_scores]

    return movies_cb.iloc[movie_indices][["title", "genres"]]


In [7]:
hybrid_recommend(user_id=1, movie_title="Toy Story (1995)", top_n=5)


,title,genres
1757,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
8695,Guardians of the Galaxy 2 (2017),Action|Adventure|Sci-Fi
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy


**Conclusion**

Hybrid filtering improves recommendation quality by combining content similarity and user behavior patterns. This approach delivers more balanced, accurate, and personalized movie recommendations compared to using a single method.